In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
import sys

In [2]:
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
llm_model_name = "llama-3.3-70b-versatile"

# mapper_llm = ChatGoogleGenerativeAI(
#     google_api_key=GOOGLE_API_KEY,
#     model = "gemini-flash-latest",
#     temperature=0
# )

mapper_llm = ChatGroq(
    model=llm_model_name,
    api_key=GROQ_API_KEY,
    temperature=0
)
mapper_llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B463F1D490>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B465051150>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [26]:
llm = ChatGoogleGenerativeAI(
    google_api_key=GOOGLE_API_KEY,
    temperature=0,
    model='gemini-2.5-flash'
)

llm.invoke('hello')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019df6a7-8c7a-7e80-9203-9cc26833aa04-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 30, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 21}})

In [ ]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)


Project root: c:\Users\beast\Documents\GEN_AI_Projects\multi-agent-security-auditor


In [4]:
from server.core.prompts import ROUTER_PROMPT,REPORTER_PROMPT,MAPPER_PROMPT,VERIFIER_PROMPT,ATTACKER_PROMPT,ALIGNER_PROMPT

In [5]:
code = """
#include <iostream>
#include <fstream>
#include <unistd.h>
#include <string>

void update_config(std::string user_provided_filename, std::string data) {
    std::string path = "/tmp/app_configs/" + user_provided_filename;

    if (access(path.c_str(), F_OK) != -1) {
        std::cout << "File exists. Updating..." << std::endl;

        std::ofstream outfile;
        outfile.open(path, std::ios_base::app);
        outfile << data;
        outfile.close();
    } else {
        std::cout << "Access denied or file missing." << std::endl;
    }
}

int main() {
    update_config("user.conf", "new_setting=true");
    return 0;
}
"""

In [6]:
mapper_chain = ChatPromptTemplate.from_template(MAPPER_PROMPT) | mapper_llm | StrOutputParser()

In [7]:
response = mapper_chain.invoke({'current_code':code})

In [8]:
response

'**Code Topology Mapping Report**\n\n### 1. Entry Points (Sources)\n\n* The `update_config` function takes two parameters: `user_provided_filename` and `data`, which can be considered as entry points for external data.\n* In the `main` function, the `update_config` function is called with hardcoded values, but in a real-world scenario, these values could be provided by an external source, such as user input or a configuration file.\n\n### 2. Sensitive Destinations (Sinks)\n\n* The `std::ofstream` object `outfile` is used to write data to a file, which is a sensitive destination.\n* The `access` function is used to check if a file exists, which can be considered a sensitive operation as it interacts with the file system.\n\n### 3. Data Flow\n\n* The `user_provided_filename` parameter is used to construct the file path, which is then passed to the `access` function to check if the file exists.\n* If the file exists, the `data` parameter is appended to the file using the `std::ofstream` o

In [9]:
attacker_chain = ChatPromptTemplate.from_template(ATTACKER_PROMPT) | mapper_llm | StrOutputParser()

In [10]:
attacker_response = attacker_chain.invoke({'current_code':code,'mapping_report':response})

In [11]:
import json, re; 
clean_json = json.loads(re.sub(r",\s*([\]}])", r"\1", attacker_response.strip()))
clean_json

[{'vulnerability_type': 'Path Traversal',
  'target_line_or_function': 'std::string path = "/tmp/app_configs/" + user_provided_filename;',
  'severity': 'High',
  'hypothesis': 'If I pass "../../../../etc/passwd" into the user_provided_filename entry point, then the contents of /etc/passwd will be accessible because the data reaches the access function unescaped.',
  'suggested_payload': '../../../../etc/passwd'},
 {'vulnerability_type': 'Arbitrary File Write',
  'target_line_or_function': 'std::ofstream outfile; outfile.open(path, std::ios_base::app);',
  'severity': 'Critical',
  'hypothesis': 'If I pass "/etc/passwd" into the user_provided_filename entry point and "malicious_data" into the data entry point, then /etc/passwd will be overwritten with malicious_data because the data reaches the std::ofstream object unescaped.',
  'suggested_payload': '/etc/passwd'},
 {'vulnerability_type': 'Directory Traversal',
  'target_line_or_function': 'std::string path = "/tmp/app_configs/" + use

In [12]:
verifier_chain = ChatPromptTemplate.from_template(VERIFIER_PROMPT) | mapper_llm | StrOutputParser()

In [13]:
verifier_responses=[]

In [14]:
verifier_response = verifier_chain.invoke({'current_code':code,'latest_vulnerability':attacker_response})

In [15]:
print(verifier_response)

```python
import os
import unittest.mock as mock

def update_config(user_provided_filename, data):
    path = "/tmp/app_configs/" + user_provided_filename

    try:
        if os.path.exists(path):
            print("File exists. Updating...")
            with open(path, 'a') as outfile:
                outfile.write(data)
        else:
            print("Access denied or file missing.")
    except Exception as e:
        print(f"An error occurred: {e}")

def test_path_traversal():
    print("--- Testing Path Traversal ---")
    payload = "../../../../etc/passwd"
    path = "/tmp/app_configs/" + payload
    try:
        with open(path, 'r') as file:
            contents = file.read()
            if "root:" in contents:
                print("VERIFICATION SUCCESS: Path Traversal")
            else:
                print("VERIFICATION FAILED")
    except Exception as e:
        print(f"An error occurred: {e}")

def test_arbitrary_file_write():
    print("--- Testing Arbitrary File Write 

In [16]:
import re

def clean_llm_code(raw_output: str) -> str:
    """
    Removes Markdown code fences and extracts only the raw Python code.
    """
    # Regex to find content inside ```python ... ``` or ``` ... ```
    # It captures the group between the backticks
    pattern = r"```(?:python|py)?\s*(.*?)\s*```"
    
    # Try to find a match
    match = re.search(pattern, raw_output, re.DOTALL | re.IGNORECASE)
    
    if match:
        # Return the inner code block
        return match.group(1).strip()
    
    # If no backticks were found, just return the stripped raw string
    # (Sometimes the LLM actually follows instructions and gives raw text)
    return raw_output.strip()

In [17]:
print(clean_llm_code(verifier_response))

import os
import unittest.mock as mock

def update_config(user_provided_filename, data):
    path = "/tmp/app_configs/" + user_provided_filename

    try:
        if os.path.exists(path):
            print("File exists. Updating...")
            with open(path, 'a') as outfile:
                outfile.write(data)
        else:
            print("Access denied or file missing.")
    except Exception as e:
        print(f"An error occurred: {e}")

def test_path_traversal():
    print("--- Testing Path Traversal ---")
    payload = "../../../../etc/passwd"
    path = "/tmp/app_configs/" + payload
    try:
        with open(path, 'r') as file:
            contents = file.read()
            if "root:" in contents:
                print("VERIFICATION SUCCESS: Path Traversal")
            else:
                print("VERIFICATION FAILED")
    except Exception as e:
        print(f"An error occurred: {e}")

def test_arbitrary_file_write():
    print("--- Testing Arbitrary File Write ---")
    

In [18]:
from server.tools.code_executor import execute_code_in_sandbox

sandbox_response = execute_code_in_sandbox(clean_llm_code(verifier_response))
sandbox_response

{'stdout': "--- Testing Path Traversal ---\nAn error occurred: [Errno 2] No such file or directory: '/tmp/app_configs/../../../../etc/passwd'\n--- Testing Arbitrary File Write ---\nAccess denied or file missing.\nAn error occurred: [Errno 2] No such file or directory: '/tmp/test_arbitrary_file_write'\n--- Testing Directory Traversal ---\nAn error occurred: [Errno 2] No such file or directory: '/tmp/app_configs/../sensitive_data'\n--- Testing Unrestricted File Upload ---\nAccess denied or file missing.\nAn error occurred: [Errno 2] No such file or directory: '/tmp/test_unrestricted_file_upload'\n",
 'stderr': '',
 'exit_code': 0}

In [19]:
reporter_chain = ChatPromptTemplate.from_template(REPORTER_PROMPT) | mapper_llm | StrOutputParser()

In [20]:
reporter_response = reporter_chain.invoke({'verification_logs':sandbox_response,'vulnerabilities_logs':verifier_response,'current_code':code})

In [21]:
print(reporter_response)

**Audit Findings Summary**

The recent audit has identified potential security vulnerabilities in the `update_config` function of the provided C++ source code. The vulnerabilities are related to path traversal, arbitrary file write, directory traversal, and unrestricted file upload.

**Vulnerability Details**

1. **Path Traversal**: The `update_config` function is vulnerable to path traversal attacks. An attacker can provide a malicious `user_provided_filename` that traverses the directory hierarchy, potentially allowing access to sensitive files.
2. **Arbitrary File Write**: The function is also vulnerable to arbitrary file write attacks. An attacker can provide a malicious `user_provided_filename` that writes data to an arbitrary file on the system.
3. **Directory Traversal**: The function is vulnerable to directory traversal attacks. An attacker can provide a malicious `user_provided_filename` that traverses the directory hierarchy, potentially allowing access to sensitive directori

In [22]:
aligner_chain = ChatPromptTemplate.from_template(ALIGNER_PROMPT) | mapper_llm | StrOutputParser()

In [23]:
aligner_response = aligner_chain.invoke({'current_code':code,'intermediate_report':reporter_response})


In [24]:
print(aligner_response)

### 🛡️ Executive Summary
The provided C++ source code contains several security vulnerabilities, including path traversal, arbitrary file write, directory traversal, and unrestricted file upload. These vulnerabilities can be exploited by an attacker to access sensitive files, write data to arbitrary files, traverse the directory hierarchy, and upload files to arbitrary locations. This report provides a deep-dive analysis of the vulnerabilities, explains the root cause, and offers a comprehensive remediation plan, including a primary fix and an alternative fix.

### 🔍 Technical Deep-Dive: Path Traversal and Arbitrary File Write
- **The Root Cause:** The `update_config` function is vulnerable to path traversal and arbitrary file write attacks due to the lack of input validation and sanitization. The `user_provided_filename` parameter is directly concatenated with the `/tmp/app_configs/` directory path, allowing an attacker to provide a malicious filename that traverses the directory hier

In [27]:
from langgraph.graph import START,END,StateGraph
from server.graph.state import AuditState
from server.graph.nodes.router import router , route_after_router
from server.graph.nodes.mapper import mapper
from server.graph.nodes.attacker import attacker
from server.graph.nodes.verifier import verifier
from server.graph.nodes.reporter import reporter
from server.graph.nodes.aligner import aligner
from server.graph.nodes.assistant import assistant
from server.graph.edges import continue_to_verification
from functools import partial

import os
from dotenv import load_dotenv
load_dotenv()
from server.core.llms import groq_with_fallback

from server.logging.logger import logging
from server.exception.exception import CustomException
import sys

ImportError: cannot import name 'ASSISTANT_PROMPT' from 'server.core.prompts' (c:\Users\beast\Documents\GEN_AI_Projects\multi-agent-security-auditor\server\core\prompts.py)